# 04 - Modelling: Classical (v2)

Trains and tunes three tree-based classifiers on the 20-feature v2 schema: Random Forest, XGBoost, and LightGBM (new candidate, not in v1; LightGBM is also what EMBER's own published baseline uses, so it is a natural fit to compare against here). Reloads `dataset_pe_v2_full.csv` directly and reproduces the same train/validation split `02_data_preparation.ipynb` showed (fixed `RANDOM_STATE`, EMBER's own `train`/`test` boundary kept, 85/15 stratified split of `train` for validation), same reproducibility approach as `legacy_v1` (no split files saved to disk, every notebook derives the identical rows on its own). Never touches the `test` rows.

**Not executed in this sandbox** (no scikit-learn/xgboost/lightgbm installed here, no network access to install them). Run locally: `Kernel -> Restart & Run All`.

In [1]:
# Standard library
import sys

# Third-party
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

sys.path.append("../src")
from constants import ORDER_OF_FEATURES, LABEL_COLUMN, RANDOM_STATE

pd.set_option("display.max_columns", None)

## Load and split

Same split logic as `02_data_preparation.ipynb`'s reference cell: EMBER's own `train`/`test` boundary as the outer split, an 85/15 stratified split of `train` for validation (after the compute-constrained training-pool downsample noted below). Fixed `RANDOM_STATE` means `05` and `06` reproduce these exact same rows independently, without any split file being saved to disk.

**Compute-constrained downsample of the training pool (2026-07, added after real local testing).** `GridSearchCV`'s exhaustive search over the full ~600,000-row training pool proved too slow on typical laptop hardware (each of the three grid searches trains many separate models across cross-validation folds). Downsamples `train_pool` to 75,000 rows per class (150,000 total, still roughly 7-8x larger than v1's ~19,600-row dataset) before the 85/15 train/validation split, using a fixed `RANDOM_STATE` so the reduction is reproducible. **The held-out `test` set (EMBER's own official 200,000-row split) is NOT reduced**: `06_evaluation.ipynb`'s final reported numbers still come from the complete, standard benchmark split, only the expensive tuning step works on a smaller pool. This is a disclosed, documented tradeoff for real hardware constraints, not a silent change.

In [2]:
df = pd.read_csv("../data/dataset_pe_v2_full.csv")
train_pool = df[df["OriginalSplit"] == "train"].reset_index(drop=True)

TRAIN_POOL_SAMPLE_PER_CLASS = 75000
train_pool = pd.concat([
    g.sample(n=min(TRAIN_POOL_SAMPLE_PER_CLASS, len(g)), random_state=RANDOM_STATE)
    for _, g in train_pool.groupby(LABEL_COLUMN)
]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
print("downsampled train_pool:", train_pool.shape)

train_df = pd.concat([
    g.sample(frac=0.85, random_state=RANDOM_STATE)
    for _, g in train_pool.groupby(LABEL_COLUMN)
]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
val_df = train_pool.drop(
    pd.concat([g.sample(frac=0.85, random_state=RANDOM_STATE) for _, g in train_pool.groupby(LABEL_COLUMN)]).index
).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

X_train, y_train = train_df[ORDER_OF_FEATURES], train_df[LABEL_COLUMN]
X_val, y_val = val_df[ORDER_OF_FEATURES], val_df[LABEL_COLUMN]
print("train:", X_train.shape, "val:", X_val.shape)

downsampled train_pool: (150000, 23)
train: (127500, 20) val: (22500, 20)


## Build the model pipelines

Each pipeline bundles a `StandardScaler` with its classifier, so the exact same fitted scaler is used at train and predict time (matches `app.py`'s inference path, same pattern as v1). Classes are exactly balanced already (50/50, from the source data), so no `class_weight`/`scale_pos_weight` correction is needed here, unlike v1 which had a 64/36 imbalance after cleaning.

In [3]:
rf_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(random_state=RANDOM_STATE)),
])

xgb_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss")),
])

lgbm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LGBMClassifier(random_state=RANDOM_STATE)),
])

## Tune and evaluate Random Forest

`GridSearchCV` scores on macro F1 so both classes count equally. `cv=3` (not 5, like v1) because this training set is far larger than v1's, each fold is already a large, stable sample; 5-fold would multiply an already nontrivial training time for little extra reliability.

In [4]:
rf_grid = GridSearchCV(
    rf_pipeline,
    param_grid={"model__n_estimators": [200, 400], "model__max_depth": [None, 20]},
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
)
rf_grid.fit(X_train, y_train)
rf_best = rf_grid.best_estimator_
print("best params:", rf_grid.best_params_)

best params: {'model__max_depth': None, 'model__n_estimators': 400}


In [5]:
rf_val_preds = rf_best.predict(X_val)
print(classification_report(y_val, rf_val_preds, target_names=["benign", "malicious"]))
print("ROC-AUC:", roc_auc_score(y_val, rf_best.predict_proba(X_val)[:, 1]))

              precision    recall  f1-score   support

      benign       0.94      0.97      0.95     11250
   malicious       0.97      0.94      0.95     11250

    accuracy                           0.95     22500
   macro avg       0.95      0.95      0.95     22500
weighted avg       0.95      0.95      0.95     22500

ROC-AUC: 0.9901581945679012


## Tune and evaluate XGBoost

In [6]:
xgb_grid = GridSearchCV(
    xgb_pipeline,
    param_grid={
        "model__n_estimators": [200, 400],
        "model__max_depth": [4, 8],
        "model__learning_rate": [0.05, 0.1],
    },
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
)
xgb_grid.fit(X_train, y_train)
xgb_best = xgb_grid.best_estimator_
print("best params:", xgb_grid.best_params_)

best params: {'model__learning_rate': 0.1, 'model__max_depth': 8, 'model__n_estimators': 400}


In [7]:
xgb_val_preds = xgb_best.predict(X_val)
print(classification_report(y_val, xgb_val_preds, target_names=["benign", "malicious"]))
print("ROC-AUC:", roc_auc_score(y_val, xgb_best.predict_proba(X_val)[:, 1]))

              precision    recall  f1-score   support

      benign       0.94      0.95      0.95     11250
   malicious       0.95      0.94      0.95     11250

    accuracy                           0.95     22500
   macro avg       0.95      0.95      0.95     22500
weighted avg       0.95      0.95      0.95     22500

ROC-AUC: 0.9886881856790124


## Tune and evaluate LightGBM

New candidate for v2. Included because EMBER's own published baseline (Anderson & Roth, 2018) uses LightGBM and reports the strongest published numbers on this dataset family (ROC AUC 0.999, source: arXiv:1804.04637), so it is a natural, literature-backed third option to compare against Random Forest and XGBoost rather than an arbitrary addition.

In [8]:
lgbm_grid = GridSearchCV(
    lgbm_pipeline,
    param_grid={
        "model__n_estimators": [200, 400],
        "model__num_leaves": [31, 63],
        "model__learning_rate": [0.05, 0.1],
    },
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
)
lgbm_grid.fit(X_train, y_train)
lgbm_best = lgbm_grid.best_estimator_
print("best params:", lgbm_grid.best_params_)

[LightGBM] [Info] Number of positive: 63750, number of negative: 63750
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003751 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4095
[LightGBM] [Info] Number of data points in the train set: 127500, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
best params: {'model__learning_rate': 0.1, 'model__n_estimators': 400, 'model__num_leaves': 63}


In [9]:
lgbm_val_preds = lgbm_best.predict(X_val)
print(classification_report(y_val, lgbm_val_preds, target_names=["benign", "malicious"]))
print("ROC-AUC:", roc_auc_score(y_val, lgbm_best.predict_proba(X_val)[:, 1]))

              precision    recall  f1-score   support

      benign       0.94      0.95      0.95     11250
   malicious       0.95      0.94      0.94     11250

    accuracy                           0.94     22500
   macro avg       0.94      0.94      0.94     22500
weighted avg       0.94      0.94      0.94     22500



C:\Users\darri\Ngee Ann Year 3S1\Emerging AI Trends in Cyber\EATC ASSG (CLAUDE)\EATC\EATC-Assignment-main\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\darri\Ngee Ann Year 3S1\Emerging AI Trends in Cyber\EATC ASSG (CLAUDE)\EATC\EATC-Assignment-main\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


ROC-AUC: 0.9876970469135802


## Save all three fitted pipelines

Final model choice (among RF/XGBoost/LightGBM/MLP) is made in `06_evaluation.ipynb` on the untouched test set, not here, so all three candidates are saved and carried forward rather than picking a winner on validation-set numbers alone.

In [10]:
joblib.dump(rf_best, "../models/rf_v2.joblib")
joblib.dump(xgb_best, "../models/xgb_v2.joblib")
joblib.dump(lgbm_best, "../models/lgbm_v2.joblib")
print("saved rf_v2.joblib, xgb_v2.joblib, lgbm_v2.joblib to ../models/")

saved rf_v2.joblib, xgb_v2.joblib, lgbm_v2.joblib to ../models/


## Summary

TODO (fill in after a real local run): validation macro F1 / ROC-AUC for each of the three models, and which looked strongest going into `05`/`06`. Next: `05_modelling_mlp.ipynb` builds a neural-network baseline for comparison, then `06_evaluation.ipynb` picks the final model using the held-out test set.